In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Non-Linear Distance Analysis: Kernel Fisher Discriminant Analysis (KFDA) via Nystroem (`models/train_distant_analysis.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and executes **Kernel Fisher Discriminant Analysis (KFDA)** using **Nystroem Non-Linear RBF Kernel Approximation** across the 8 core arrival triage features to project the non-linear emergency manifold and separate **`ESI 1 (Immediate Resuscitation)`** from **`NOT ESI 1 (ESI 2–5)`**.

### 🎯 Target Formulation
- **`Class 1: ESI 1 (Immediate Resuscitation)`** ($y=1$): Patients requiring immediate life-saving intervention ($5,271$ visits, $\sim 0.94\%$).
- **`Class 0: NOT ESI 1 (ESI 2–5)`** ($y=0$): Emergent, urgent, and non-urgent visits ($552,758$ visits, $\sim 99.06\%$).

### 🩺 8 Core Arrival Triage Features
1. `age`
2. `cc_breathingdifficulty`
3. `gender` (0 = Female, 1 = Male)
4. `triage_vital_hr` (Heart Rate)
5. `triage_vital_sbp` (Systolic Blood Pressure)
6. `triage_vital_dbp` (Diastolic Blood Pressure)
7. `triage_vital_rr` (Respiratory Rate)
8. `triage_vital_o2` (Oxygen Saturation - SpO2)

### 🔬 Mathematical & Non-Linear Mapping Strategy
1. **Non-Linear RBF Kernel Feature Map**: Evaluates the Gaussian RBF kernel $k(x, x') = \exp(-\gamma ||x - x'||^2)$ to capture non-linear physiological risk boundaries (e.g. U-shaped heart rate and blood pressure extremes).
2. **Nystroem Kernel Approximation (`sklearn.kernel_approximation.Nystroem`)**: Maps $\mathbb{R}^8 \to \mathbb{R}^{600}$ using $m=600$ landmark components, scaling efficiently across massive emergency cohorts.
3. **Kernel Fisher Discriminant Analysis (KFDA)**: Maximizes Rayleigh's quotient $J(w) = \frac{w^T S_B w}{w^T S_W w}$ in the kernel Hilbert space to obtain the optimal non-linear discriminant coordinate $z_1$.
4. **Before vs After Visualization**: Compares raw overlapping vital scatter plots with the non-linear KFDA manifold.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data from R, Impute & Partition
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Binary Target Formulation: 1 = ESI 1 (Immediate Resuscitation), 0 = NOT ESI 1 (ESI 2-5)
y_all = np.where(esi_all == 1, 1, 0)
LABELS = ['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']

print("=========================================================")
print("     5v_cleandf COHORT FOR DISTANCE / MANIFOLD ANALYSIS")
print("=========================================================")
print(f"Total Valid ESI Visits: {len(y_all):,}")
print(f"  * Class 0 [NOT ESI 1 (ESI 2-5)]: {np.sum(y_all == 0):,} ({np.mean(y_all == 0)*100:.2f}%)")
print(f"  * Class 1 [ESI 1 (Resuscitation)]: {np.sum(y_all == 1):,} ({np.mean(y_all == 1)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================\n")

# Stratified 70% Train / 30% Holdout Test Split
itr, ite = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)

raw_tr = raw_mat_all[itr]
raw_te = raw_mat_all[ite]
y_tr   = y_all[itr]
y_te   = y_all[ite]

# SimpleImputer + StandardScaler fitted strictly on Train
imputer = SimpleImputer(strategy='median')
X_tr_imp = imputer.fit_transform(raw_tr)
X_te_imp = imputer.transform(raw_te)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_imp)
X_te_scaled = scaler.transform(X_te_imp)

print(f"✓ Processed Shapes: Train={X_tr_scaled.shape}, Test={X_te_scaled.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Build Non-Linear Kernel Feature Space via Nystroem & Train KFDA
# ---------------------------------------------------------------------------
print("Building Non-Linear RBF Kernel Feature Map via Nystroem Approximation...")

# Nystroem Non-Linear RBF Embedding
# n_components=600 landmark samples provides high-fidelity RKHS approximation
nystroem = Nystroem(
    kernel='rbf',
    gamma=0.15,
    n_components=600,
    random_state=42,
    n_jobs=-1
)

print("Transforming feature space: X in R^8 -> phi(X) in R^600...")
Phi_train = nystroem.fit_transform(X_tr_scaled)
Phi_test  = nystroem.transform(X_te_scaled)

print(f"✓ Nystroem Feature Space Dimensions: Train={Phi_train.shape}, Test={Phi_test.shape}")

# Train Fisher Discriminant Analysis on the Non-linear Feature Space (KFDA)
print("\nSolving Kernel Fisher Discriminant Analysis (KFDA)...")
kfda = LinearDiscriminantAnalysis(n_components=1)
kfda.fit(Phi_train, y_tr)

# 1D Kernel Fisher Discriminant Scores
z1_tr = kfda.transform(Phi_train).ravel()
z1_te = kfda.transform(Phi_test).ravel()

# Linear Baseline FDA on raw 8 features for direct comparison
lda_linear = LinearDiscriminantAnalysis(n_components=1)
lda_linear.fit(X_tr_scaled, y_tr)
z1_linear_te = lda_linear.transform(X_te_scaled).ravel()

# Construct 2D Manifold: Component 1 = KFDA Axis, Component 2 = Orthogonal Kernel Principal Component
print("Extracting Orthogonal Principal Direction in Kernel Space for 2D Manifold...")
w_kfd = kfda.coef_  # (1, 600)
w_norm = w_kfd / np.linalg.norm(w_kfd)
Phi_ortho_tr = Phi_train - np.dot(Phi_train, w_norm.T) * w_norm
Phi_ortho_te = Phi_test - np.dot(Phi_test, w_norm.T) * w_norm

pca_ortho = PCA(n_components=1, random_state=42)
z2_tr = pca_ortho.fit_transform(Phi_ortho_tr).ravel()
z2_te = pca_ortho.transform(Phi_ortho_te).ravel()

# 2D Raw Baseline PCA for Before-Projection visualization
pca_raw = PCA(n_components=2, random_state=42)
X_raw_pca_te = pca_raw.fit_transform(X_te_scaled)

print("✓ 2D Manifold Coordinates Successfully Computed!")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Quantitative Distance & Separability Metrics (Linear vs KFDA)
# ---------------------------------------------------------------------------
# Compute Fisher Criterion J(w) = (mu_1 - mu_0)^2 / (var_1 + var_0)
def compute_fisher_ratio(z_scores, y_true):
    z_pos = z_scores[y_true == 1]
    z_neg = z_scores[y_true == 0]
    mu_diff_sq = (np.mean(z_pos) - np.mean(z_neg)) ** 2
    var_sum = np.var(z_pos) + np.var(z_neg)
    return mu_diff_sq / (var_sum + 1e-10)

def compute_centroid_distance(z_scores, y_true):
    z_pos = z_scores[y_true == 1]
    z_neg = z_scores[y_true == 0]
    pooled_std = np.sqrt(0.5 * (np.var(z_pos) + np.var(z_neg)))
    return np.abs(np.mean(z_pos) - np.mean(z_neg)) / (pooled_std + 1e-10)

j_linear = compute_fisher_ratio(z1_linear_te, y_te)
j_kfda   = compute_fisher_ratio(z1_te, y_te)

d_linear = compute_centroid_distance(z1_linear_te, y_te)
d_kfda   = compute_centroid_distance(z1_te, y_te)

auc_linear = roc_auc_score(y_te, z1_linear_te)
if auc_linear < 0.5:
    auc_linear = 1.0 - auc_linear

auc_kfda = roc_auc_score(y_te, z1_te)
if auc_kfda < 0.5:
    auc_kfda = 1.0 - auc_kfda

pr_auc_linear = average_precision_score(y_te, z1_linear_te if roc_auc_score(y_te, z1_linear_te) >= 0.5 else -z1_linear_te)
pr_auc_kfda   = average_precision_score(y_te, z1_te if roc_auc_score(y_te, z1_te) >= 0.5 else -z1_te)

metrics_data = [
    {
        'Projection_Method': '1. Raw Linear FDA (Baseline)',
        'Feature_Space': 'Original Input R^8',
        'Kernel': 'None (Linear)',
        'Fisher_Ratio J(w)': round(j_linear, 4),
        'Class_Centroid_Distance (Cohen d)': round(d_linear, 4),
        'ROC_AUC': round(auc_linear, 4),
        'PR_AUC': round(pr_auc_linear, 4)
    },
    {
        'Projection_Method': '2. Kernel Fisher DA (Nystroem KFDA)',
        'Feature_Space': 'Nystroem RKHS R^600',
        'Kernel': 'Non-Linear RBF (gamma=0.15)',
        'Fisher_Ratio J(w)': round(j_kfda, 4),
        'Class_Centroid_Distance (Cohen d)': round(d_kfda, 4),
        'ROC_AUC': round(auc_kfda, 4),
        'PR_AUC': round(pr_auc_kfda, 4)
    }
]

metrics_df = pd.DataFrame(metrics_data)
print("=========================================================================================================")
print("         SEPARABILITY & DISTANCE ANALYSIS: LINEAR FDA vs NYSTROEM KERNEL FDA")
print("=========================================================================================================")
print(metrics_df.to_string(index=False))
print("=========================================================================================================\n")

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'kfda_nystroem_separability_report.csv')
metrics_df.to_csv(report_file, index=False)
print(f"Separability report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: BEFORE vs AFTER Projection Scatter Plots
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

# Sample test points for visual clarity (all ESI 1 + 2500 sampled NOT ESI 1)
np.random.seed(42)
test_esi1_idx = np.where(y_te == 1)[0]
test_not_idx  = np.where(y_te == 0)[0]
test_not_sample = np.random.choice(test_not_idx, min(2500, len(test_not_idx)), replace=False)

fig, axes = plt.subplots(2, 2, figsize=(17, 14))

# Panel 1: BEFORE - Raw Vital Plane 1: Heart Rate vs Systolic BP
hr_idx = FEATURES.index('triage_vital_hr')
sbp_idx = FEATURES.index('triage_vital_sbp')

axes[0, 0].scatter(X_te_imp[test_not_sample, hr_idx], X_te_imp[test_not_sample, sbp_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (ESI 2-5)')
axes[0, 0].scatter(X_te_imp[test_esi1_idx, hr_idx], X_te_imp[test_esi1_idx, sbp_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 0].set_title('BEFORE: Raw Features (Heart Rate vs Systolic BP)\n[Heavy Class Overlap & Non-Linear Entanglement]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Heart Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_ylabel('Systolic BP (mmHg)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_xlim(30, 200)
axes[0, 0].set_ylim(50, 240)
axes[0, 0].grid(True, linestyle=':', alpha=0.4)
axes[0, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 2: BEFORE - Raw Vital Plane 2: Respiratory Rate vs SpO2
rr_idx = FEATURES.index('triage_vital_rr')
o2_idx = FEATURES.index('triage_vital_o2')

axes[0, 1].scatter(X_te_imp[test_not_sample, rr_idx], X_te_imp[test_not_sample, o2_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (ESI 2-5)')
axes[0, 1].scatter(X_te_imp[test_esi1_idx, rr_idx], X_te_imp[test_esi1_idx, o2_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 1].set_title('BEFORE: Raw Features (Respiratory Rate vs SpO2)\n[Severe Clinical Overlap in Oxygenation]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Respiratory Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_ylabel('Oxygen Saturation SpO2 (%)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_xlim(6, 45)
axes[0, 1].set_ylim(70, 100)
axes[0, 1].grid(True, linestyle=':', alpha=0.4)
axes[0, 1].legend(loc='lower left', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 3: BEFORE - 2D Linear PCA of Raw 8 Features
axes[1, 0].scatter(X_raw_pca_te[test_not_sample, 0], X_raw_pca_te[test_not_sample, 1],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (ESI 2-5)')
axes[1, 0].scatter(X_raw_pca_te[test_esi1_idx, 0], X_raw_pca_te[test_esi1_idx, 1],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[1, 0].set_title('BEFORE: 2D Linear PCA (Original 8-Feature Space)\n[Linear Projection Fails to Untangle Resuscitation Cluster]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Linear Principal Component 1', fontsize=10.5, fontweight='bold')
axes[1, 0].set_ylabel('Linear Principal Component 2', fontsize=10.5, fontweight='bold')
axes[1, 0].grid(True, linestyle=':', alpha=0.4)
axes[1, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 4: AFTER - 2D Nystroem Kernel Fisher Discriminant Manifold
axes[1, 1].scatter(z1_te[test_not_sample], z2_te[test_not_sample],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (ESI 2-5)')
axes[1, 1].scatter(z1_te[test_esi1_idx], z2_te[test_esi1_idx],
                   c='#d62728', alpha=0.85, s=40, edgecolors='black', linewidth=0.6, label='ESI 1 (Resuscitation)')
# Draw decision separating boundary on KFDA axis
kfd_threshold = (np.mean(z1_te[test_esi1_idx]) + np.mean(z1_te[test_not_idx])) / 2.0
axes[1, 1].axvline(x=kfd_threshold, color='black', linestyle='--', linewidth=2.0, label=f'KFDA Separator (z1 = {kfd_threshold:.2f})')

axes[1, 1].set_title(f'AFTER: 2D Nystroem Kernel Fisher Discriminant Manifold (KFDA)\n[Non-Linear RBF Kernel Space (m=600) | Fisher J(w) = {j_kfda:.2f}]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Component 1: Kernel Fisher Discriminant Axis (z1)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_ylabel('Component 2: Orthogonal Kernel Variance Axis (z2)', fontsize=10.5, fontweight='bold')
axes[1, 1].grid(True, linestyle=':', alpha=0.4)
axes[1, 1].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

plt.suptitle('Non-Linear Distance Analysis: Kernel Fisher Discriminant Analysis (KFDA via Nystroem)\nBEFORE vs AFTER Feature Projection (ESI 1 vs NOT ESI 1)', fontsize=14.5, fontweight='bold', y=0.995)
plt.tight_layout()

scatter_comparison_file = os.path.join(plots_dir, 'kfda_nystroem_before_after_scatter.png')
plt.savefig(scatter_comparison_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'distant_analysis', 'kfda_nystroem_before_after_scatter.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'kfda_nystroem_before_after_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Before/After Scatter comparison saved to: {scatter_comparison_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 1D Kernel Fisher Discriminant Density Distribution (KDE / Histogram)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Linear FDA Distribution (Baseline)
sns.kdeplot(z1_linear_te[y_te == 0], ax=axes[0], color='#1f77b4', fill=True, alpha=0.3, linewidth=2.0, label='NOT ESI 1 (ESI 2-5)')
sns.kdeplot(z1_linear_te[y_te == 1], ax=axes[0], color='#d62728', fill=True, alpha=0.3, linewidth=2.0, label='ESI 1 (Resuscitation)')
axes[0].set_title(f'Linear FDA 1D Projection (Baseline)\nFisher J(w) = {j_linear:.4f} | ROC-AUC = {auc_linear:.4f}', fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel('Linear Discriminant Projection Score (z1)', fontsize=10.5, fontweight='bold')
axes[0].set_ylabel('Probability Density', fontsize=10.5, fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.4)
axes[0].legend(loc='upper right', fontsize=10)

# Panel 2: Nystroem Kernel FDA Distribution
sns.kdeplot(z1_te[y_te == 0], ax=axes[1], color='#1f77b4', fill=True, alpha=0.3, linewidth=2.0, label='NOT ESI 1 (ESI 2-5)')
sns.kdeplot(z1_te[y_te == 1], ax=axes[1], color='#d62728', fill=True, alpha=0.3, linewidth=2.0, label='ESI 1 (Resuscitation)')
axes[1].axvline(x=kfd_threshold, color='black', linestyle='--', linewidth=1.8, label=f'Optimal Separator ({kfd_threshold:.2f})')
axes[1].set_title(f'Nystroem Kernel FDA (KFDA) 1D Projection\nFisher J(w) = {j_kfda:.4f} | ROC-AUC = {auc_kfda:.4f}', fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel('Kernel Fisher Discriminant Score (z1)', fontsize=10.5, fontweight='bold')
axes[1].set_ylabel('Probability Density', fontsize=10.5, fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.4)
axes[1].legend(loc='upper right', fontsize=10)

plt.tight_layout()
kfd_density_file = os.path.join(plots_dir, 'kfda_1d_density_distribution.png')
plt.savefig(kfd_density_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'distant_analysis', 'kfda_1d_density_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"1D Density distribution saved to: {kfd_density_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production KFDA Transformation Pipeline & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

kfda_bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'nystroem': nystroem,
    'kfda_model': kfda,
    'pca_ortho': pca_ortho,
    'w_norm': w_norm,
    'features': FEATURES,
    'labels': LABELS,
    'threshold': kfd_threshold
}

bundle_file = os.path.join(deploy_dir, 'kfda_nystroem_projection_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(kfda_bundle, f)

manifest = dict(
    pipeline_type='Kernel_Fisher_Discriminant_Analysis_KFDA',
    kernel_method='Nystroem_RBF_Kernel_Approximation',
    kernel_gamma=0.15,
    nystroem_components=600,
    dataset='5v_cleandf_RData',
    features=FEATURES,
    target='ESI1_vs_NOT_ESI1',
    total_samples=len(y_all),
    n_esi1=int(np.sum(y_all == 1)),
    n_not_esi1=int(np.sum(y_all == 0)),
    metrics=metrics_data,
    fisher_ratio_gain=round(j_kfda / (j_linear + 1e-10), 2)
)

manifest_file = os.path.join(deploy_dir, 'kfda_nystroem_projection_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported KFDA Bundle  : {bundle_file}")
print(f"✓ Exported KFDA Manifest: {manifest_file}")